<h2>MMLD NexGrid Data Dump Parsing Playground</h1><br>
Tools and scripts to parse NexGrid CSV output as needed for successful delivery to the MS SQL Server. 

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy import text   
from sqlalchemy import event
import urllib
import os
import requests
import httpx
from pyproj import Transformer
from shapely import wkb

<h3>1. Parsing NexGrid Data</h3>
<i>Note</i>: This data will be in the form of a .csv file. Test data initially comes from a .txt file. Ideally, the file name will include some information like interval (month, day, year), and location (town, feeder, transformer, etc.) that can be parsed out as metadata.

In [6]:
my_dir = "C:/Users/bknight/OneDrive - Marblehead Municipal Light Plant/Documents/github/mmld"
if os.getcwd() != my_dir:
    os.chdir(my_dir) # for testing purposes
    print(f"Changed working directory to {my_dir}")
print(f"Current working directory: {os.getcwd()}")
print(os.listdir())
if "data" in os.listdir():
    print("Data directory found.")
    print(os.listdir("data"))

Changed working directory to C:/Users/bknight/OneDrive - Marblehead Municipal Light Plant/Documents/github/mmld
Current working directory: C:\Users\bknight\OneDrive - Marblehead Municipal Light Plant\Documents\github\mmld
['.git', 'data', 'README.md', 'requirements.txt']
Data directory found.
['elec_hour_20260608.csv', 'nexgrid.txt']


In [92]:
# Read the data in
nexgrid_df = pd.read_csv("data/elec_hour_20260608.csv").rename(columns={"Serial": "Meter_ID"})
nexgrid_df.columns = nexgrid_df.columns.str.lower()
print(f"Data read in successfully. DataFrame shape: {nexgrid_df.shape}\nFirst 5 rows:")
display(nexgrid_df.head())
print(f"\nDataFrame description:")
display(nexgrid_df.describe())
print(f"\nDataFrame info:")
display(nexgrid_df.info())

Data read in successfully. DataFrame shape: (246822, 22)
First 5 rows:


,meter_id,multiplier,time,kwh,kwh usage,received kwh,received kwh usage,peak kw,kvarh,kvarh usage,...,kvah usage,peak kvah,custom register,custom ext,custom register 2,custom ext 2,tou a,tou b,tou c,tou d
0,54442533,1.0,2026/6/8 12:00:00 AM,11252.6587,0.1907,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,471.149898,2050.860558,0.0,0.0
1,54442292,1.0,2026/6/8 12:00:00 AM,2120.8156,0.4598,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2696.139699,12919.499791,0.0,0.0
2,53374650,1.0,2026/6/8 12:00:00 AM,98538.4475,0.4963,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2185.674192,9132.332912,0.0,0.0
3,50084405,1.0,2026/6/8 12:00:00 AM,64848.3849,0.6426,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1380.043339,6930.786196,0.0,0.0
4,52857677,1.0,2026/6/8 12:00:00 AM,39560.4159,0.1927,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,37.816615,196.125984,0.0,0.0



DataFrame description:


,meter_id,multiplier,kwh,kwh usage,received kwh,received kwh usage,peak kw,kvarh,kvarh usage,peak kvarh,...,kvah usage,peak kvah,custom register,custom ext,custom register 2,custom ext 2,tou a,tou b,tou c,tou d
count,2.468220e+05,246822.000000,2.366090e+05,246658.000000,233486.000000,230693.000000,13458.000000,3021.0,1449.0,2565.0,...,1846.0,2786.0,3258.0,3258.000000,3258.0,3258.0,184237.000000,184237.000000,184237.0,184237.0
mean,5.364489e+07,1.645522,5.596696e+04,1.020607,223.096839,0.013397,8.485841,0.0,0.0,0.0,...,0.0,0.0,0.0,0.787293,0.0,0.0,624.556930,3315.370387,0.0,0.0
std,1.058042e+07,9.482581,2.299890e+05,3.937617,2889.758278,0.304818,18.582072,0.0,0.0,0.0,...,0.0,0.0,0.0,0.409285,0.0,0.0,975.173928,5312.913863,0.0,0.0
min,1.175365e+07,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
25%,5.285752e+07,1.000000,2.260030e+04,0.208100,0.000000,0.000000,2.310000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,73.452417,378.811875,0.0,0.0
50%,5.337374e+07,1.000000,4.581027e+04,0.488000,0.000000,0.000000,5.708000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,312.838654,1616.990177,0.0,0.0
75%,5.444169e+07,1.000000,7.196716e+04,1.016400,0.000000,0.000000,8.520000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,742.371191,3875.803377,0.0,0.0
max,9.970000e+07,600.000000,2.018912e+07,601.200000,85366.890800,31.919500,349.200000,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,0.0,0.0,23930.434950,140762.526838,0.0,0.0



DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 246822 entries, 0 to 246821
Data columns (total 22 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   meter_id            246822 non-null  int64  
 1   multiplier          246822 non-null  float64
 2   time                246822 non-null  str    
 3   kwh                 236609 non-null  float64
 4   kwh usage           246658 non-null  float64
 5   received kwh        233486 non-null  float64
 6   received kwh usage  230693 non-null  float64
 7   peak kw             13458 non-null   float64
 8   kvarh               3021 non-null    float64
 9   kvarh usage         1449 non-null    float64
 10  peak kvarh          2565 non-null    float64
 11  kvah                3434 non-null    float64
 12  kvah usage          1846 non-null    float64
 13  peak kvah           2786 non-null    float64
 14  custom register     3258 non-null    float64
 15  custom ext          3258 non

None

In [96]:
# Get unique meter IDs
def get_unique_meter_ids(df):
    return df["meter_id"].unique()

In [93]:
# Get percentage null of each column
def remove_unused_cols(df):
    percent_missing = df.isnull().sum() * 100 / len(df)
    missing_value_pct = pd.DataFrame({"column_name": df.columns,
                                    "percent_missing": percent_missing})
    missing_value_pct.sort_values('percent_missing', inplace=True)
    display(missing_value_pct)

    removed_cols = []
    print("Removing unused columns...")
    for idx, row in missing_value_pct.iterrows():
        # Remove columns that are more than 90% empty for now and list them
        if row["percent_missing"] > 90.0:
            column = row["column_name"]
            df.drop([column], axis=1, inplace=True)
            removed_cols += [column]
    #print(*items, sep=", ")   
    print(f"The following rows were removed:{", ".join(removed_cols)}")   
    return df


<h3>2. Connect to SQL database & Start SQLAlchemy engine

In [ ]:
# Connect to MS SQL database
def connect_to_db(conn_str):
    try:
        return pyodbc.connect(conn_str)
    except Exception as e:
        print(f"Database connectioned failed!\n{e}")
    print("Database connection established successfully!")

# Create a base connection string
def create_conn_str(server, database):
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={server};" # MMLDAPP03
        f"DATABASE={database};" # MMLDGIS
        "Trusted_Connection=yes;"
        "MARS_Connection=yes;"
    )

# Parse a disparate connection string
def encode_conn_url(conn_str):
    return urllib.parse.quote_plus(conn_str)

# Connect to SQL Alchemy engine
def connect_to_engine(conn_str):
    params = encode_conn_url(conn_str)
    return create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Convert a shape column to EWKB data for GEOS compatibility
def shapes_to_ewkb(engine, database):
    # Get the underlying pyodbc connection from the engine to use the hierarchy ID converter
    # A regular connection will not support low level conversions
    with engine.connect() as conn:
        # Apply converter for hierarchyid if needed
        dbapi_conn = conn.connection.dbapi_connection # Exposes low level pyodbc driver without closing SQLalchemy driver
        dbapi_conn.add_output_converter(-151, lambda value: str(value))
        
        # Get the EWKB binary data
        ewkb_data = pd.read_sql(f"""
            SELECT 
                meter_id,
                SHAPE.STAsBinary() AS ewkb_data
            FROM [{database}].[dbo].[meternxt]
            WHERE meter_id IS NOT NULL
                AND ISNUMERIC(meter_id) = 1
        """, conn)

        return ewkb_data
    
# Convert WKB hex type 345 to EWKB and get coordinates for GEOS
def get_coords_from_ewkb(ewkb_bytes):
    if ewkb_bytes is None:
        return None, None

    # Define a transformer for projecting EPSG:2249/NAD83 to EPSG:4326 to get Lat/Lon in degrees, not US survey feet
    # EPSG:2249 (NAD83 / Massachusetts Mainland ftUS) -> EPSG:4326 (WGS84 Lat/Lon)
    # always_xy=True ensures input is (X, Y) and output is (Lon, Lat), standard to Cartesian systems
    cartesian_transformer = Transformer.from_crs("EPSG:2249", "EPSG:4326", always_xy=True)

    # Load WKB (Shapely reads as X, Y -> Lon, Lat in projected space)
    geom = wkb.loads(ewkb_bytes)

    if geom.geom_type == 'Point':
        # Transform from Feet (X, Y) to Degrees (Lon, Lat)
        lon, lat = cartesian_transformer.transform(geom.x, geom.y)
        return lat, lon # Return as Lat, Lon for readability
    else:
        # For polygons/lines, transform all coordinates
        from shapely.ops import transform
        geom_4326 = transform(cartesian_transformer.transform, geom)
        # Return centroid
        return geom_4326.centroid.y, geom_4326.centroid.x

In [50]:
# Connect to database
SERVER_NAME = "MMLDAPP03"
DATABASE_NAME = "MMLDGIS"
conn_str = create_conn_str(SERVER_NAME, DATABASE_NAME)
connection = connect_to_db(conn_str)
engine = connect_to_engine(conn_str) # Create SQL Alchemy engine

In [85]:
# Apply to dataframe
coords = shapes_to_ewkb(engine, DATABASE_NAME)

# Make sure meter_ids are np.int64 before merging
if type(coords["meter_id"].iloc[0,]) is not np.int64:
    coords["meter_id"] = coords["meter_id"].astype(np.int64)

coords[['latitude', 'longitude']] = coords['ewkb_data'].apply(
    lambda x: pd.Series(get_coords_from_ewkb(x))
)

coords.drop(["ewkb_data"], axis=1)

print(f"Successfully converted {len(coords)} shapes to (X, Y) coordinate pairs.")
print(coords[['meter_id', 'latitude', 'longitude']].head())

Successfully converted 11726 shapes to (X, Y) coordinate pairs.
   meter_id   latitude  longitude
0  50084651  42.501582 -70.851822
1  50084657  42.493610 -70.865143
2  50084659  42.503837 -70.849594
3  50084660  42.510322 -70.853394
4  50084663  42.498160 -70.854508


In [64]:
def merge_new_reads(new_reads, reads_db, engine):
    return new_reads.merge(reads_db, on="meter_id", how="inner")

In [65]:
type(nexgrid_df["meter_id"].iloc[0,])

numpy.int64

In [94]:
merged_df = merge_new_reads(nexgrid_df, coords, engine)

In [95]:
display(remove_unused_cols(merged_df))

,column_name,percent_missing
meter_id,meter_id,0.000000
multiplier,multiplier,0.000000
time,time,0.000000
kwh usage,kwh usage,0.057630
kwh,kwh,3.822037
received kwh,received kwh,4.945270
received kwh usage,received kwh usage,5.983327
ewkb_data,ewkb_data,11.882566
latitude,latitude,11.882566
longitude,longitude,11.882566


Removing unused columns...
The following rows were removed:peak kw, kvah, custom ext 2, custom register, custom ext, custom register 2, kvarh, peak kvah, peak kvarh, kvah usage, kvarh usage


,meter_id,multiplier,time,kwh,kwh usage,received kwh,received kwh usage,tou a,tou b,tou c,tou d,ewkb_data,latitude,longitude
0,54442533,1.0,2026/6/8 12:00:00 AM,11252.6587,0.1907,0.0,0.0,471.149898,2050.860558,0.0,0.0,b'\x01\x01\x00\x00\x00\x02/^;\xcaM)A\x06\x85T\...,42.509444,-70.858421
1,54442292,1.0,2026/6/8 12:00:00 AM,2120.8156,0.4598,0.0,0.0,2696.139699,12919.499791,0.0,0.0,b'\x01\x01\x00\x00\x007_\x81\xafK7)A\xa8\xa1\x...,42.501563,-70.869179
2,53374650,1.0,2026/6/8 12:00:00 AM,98538.4475,0.4963,0.0,0.0,2185.674192,9132.332912,0.0,0.0,b'\x01\x01\x00\x00\x00\x19Q]aCD)A\xf8B\xe6\x05...,42.514579,-70.862892
3,50084405,1.0,2026/6/8 12:00:00 AM,64848.3849,0.6426,0.0,0.0,1380.043339,6930.786196,0.0,0.0,"b'\x01\x01\x00\x00\x00\xf3\xcb\x00\x12""C)AFq\x...",42.498931,-70.863586
4,50084405,1.0,2026/6/8 12:00:00 AM,64848.3849,0.6426,0.0,0.0,1380.043339,6930.786196,0.0,0.0,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275895,53373716,1.0,2026/6/8 11:00:00 PM,56596.5114,0.0000,0.0,0.0,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x007~\xeb\x08\xc1Q)A.\x9cx\...,42.496970,-70.856667
275896,50084121,1.0,2026/6/8 11:00:00 PM,10445.7890,0.0000,0.0,0.0,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\xea)\xd7NvY)AQ\xe4\x18\...,42.504020,-70.852936
275897,50084121,1.0,2026/6/8 11:00:00 PM,10445.7890,0.0000,0.0,0.0,NaN,NaN,NaN,NaN,None,NaN,NaN
275898,50088510,1.0,2026/6/8 11:00:00 PM,30124.5421,0.1797,NaN,NaN,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\x806=\xb7Bi)A\xe4L\xd8\...,42.509204,-70.845383


In [98]:
merged_df["meter_id"].value_counts()

meter_id
50084405    48
50084808    48
50085742    48
50085277    48
50084043    48
            ..
50088694    16
50088621    16
58949435     9
58949525     7
53373220     6
Name: count, Length: 10155, dtype: int64

In [ ]:
# List tables in the database -- ctrl + / to create a multiline comment
# if "cursor" in locals():
#     try:
#         cursor.fetchall()
#         cursor.close()
#     except Exception as e:
#         print(f"Error closing cursor: {e}")